In [1]:
import os
import pandas as pd


from io import BytesIO
from dotenv import load_dotenv
from pymongo import MongoClient
from azure.storage.filedatalake import DataLakeServiceClient


# 1. CARREGAR CREDENCIAIS

load_dotenv()

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")


# 2. CONECTAR AO MONGODB

mongo_uri = os.getenv("MONGO_URI")
mongo_database = os.getenv("MONGO_DATABASE")

if not mongo_uri or not mongo_database:
    raise ValueError("Configurações do MongoDB não encontradas no .env")

cliente = MongoClient(mongo_uri)

db = cliente[mongo_database]

colecao = db["eventos_site"]

print("Conexão com MongoDB realizada com sucesso!")


# 3. CONECTAR AO AZURE DATA LAKE

datalake_service = DataLakeServiceClient.from_connection_string(
    connection_string
)

file_system = datalake_service.get_file_system_client(
    file_system="bronze"
)

diretorio = file_system.get_directory_client(
    "mongodb/eventos_site"
)

print("Conexão com Azure Data Lake realizada com sucesso!")


# 4. DEFINIR COLEÇÕES

colecoes = [
    "eventos_site"
]


# 5. EXTRAIR E ENVIAR PARA BRONZE

try:

    for nome_colecao in colecoes:

        print(f"\nProcessando: {nome_colecao}")

        # Extrair dados do MongoDB
        colecao = db[nome_colecao]

        documentos = list(colecao.find())

        df = pd.DataFrame(documentos)

        print(f"Registros extraídos: {len(df)}")

        if df.empty:
            raise ValueError(f"A coleção {nome_colecao} está vazia.")

        # Converter ObjectId para string
        df["_id"] = df["_id"].astype(str)

        # Converter DataFrame para Parquet em memória
        buffer = BytesIO()

        df.to_parquet(
            buffer,
            index=False,
            engine="pyarrow"
        )

        # Definir arquivo no Azure
        arquivo = diretorio.get_file_client(
            f"{nome_colecao}.parquet"
        )

        # Enviar para Bronze
        arquivo.upload_data(
            buffer.getvalue(),
            overwrite=True
        )

        print(f"{nome_colecao}.parquet enviado para Bronze.")

finally:

    # 6. FINALIZAÇÃO
    cliente.close()

print("\nIngestão concluída com sucesso!")

Conexão com MongoDB realizada com sucesso!
Conexão com Azure Data Lake realizada com sucesso!

Processando: eventos_site
Registros extraídos: 539868
eventos_site.parquet enviado para Bronze.

Ingestão concluída com sucesso!
